# Compare IPA: avoid-step (baseline) vs floor asymptote

Overlays IPA-vs-P% from the **working `approach_1_avoid_step`** (power-law asymptote A) against
the **floor-asymptote** variant, for every batch size.

It reads each method's **saved summary CSV** from disk — it does NOT recompute anything. So when
you change/re-run the avoid-step notebook (its `ipa_summary_*.csv` is rewritten), just re-run this
notebook and the comparison reflects the new numbers automatically.

If you move a notebook's output location, update `AVOID_STEP_SUMMARY` / `FLOOR_SUMMARY` below.

In [7]:
# === Cell 1 — Load both summary CSVs (the actual saved outputs of each method) ===
import os
import numpy as np
import pandas as pd

# --- Paths to each method's saved summary (edit if you relocate outputs) ---
AVOID_STEP_SUMMARY = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step\ipa_summary_approach_1_avoid_step.csv"
FLOOR_SUMMARY      = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor\ipa_summary_approach_1_avoid_step_floor.csv"
OUT_DIR            = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor"

BATCH_SIZES = [64, 1024, 60000]

for tag, path in [("avoid-step", AVOID_STEP_SUMMARY), ("floor", FLOOR_SUMMARY)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"[{tag}] summary not found:\n  {path}\n"
                                f"Run that method's compute notebook first, or fix the path above.")

avoid_df = pd.read_csv(AVOID_STEP_SUMMARY); avoid_df.columns = avoid_df.columns.str.strip()
floor_df = pd.read_csv(FLOOR_SUMMARY);      floor_df.columns = floor_df.columns.str.strip()
print("avoid-step summary:", AVOID_STEP_SUMMARY)
print("floor      summary:", FLOOR_SUMMARY)
print(f"Loaded {len(avoid_df)} / {len(floor_df)} pruning rows.")

avoid-step summary: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step\ipa_summary_approach_1_avoid_step.csv
floor      summary: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor\ipa_summary_approach_1_avoid_step_floor.csv
Loaded 19 / 19 pruning rows.


In [8]:
# === Cell 2 — Overlay IPA vs P%: avoid-step (dashed) vs floor (solid), per batch size ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

fig, axes = plt.subplots(1, len(BATCH_SIZES), figsize=(6 * len(BATCH_SIZES), 5), sharex=True)
if len(BATCH_SIZES) == 1:
    axes = [axes]

for ax, bs in zip(axes, BATCH_SIZES):
    col = f"IPA_Avg_{bs}"
    a = avoid_df.dropna(subset=[col]) if col in avoid_df.columns else avoid_df.iloc[0:0]
    f = floor_df.dropna(subset=[col]) if col in floor_df.columns else floor_df.iloc[0:0]
    ax.plot(a["P%"].values, a[col].values, "o--", color="#999999", ms=5, lw=1.6,
            label="avoid-step (power-law A)")
    ax.plot(f["P%"].values, f[col].values, "o-", color=BS_COLOR.get(bs, "#1f77b4"), ms=5, lw=2,
            label="floor asymptote")
    ax.set_title(f"BS={bs}")
    ax.set_xlabel("Pruning Percentage (%)")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False, fontsize=10)
axes[0].set_ylabel("IPA")
fig.suptitle("IPA vs Pruning  —  avoid-step (dashed grey) vs floor asymptote (solid)", fontsize=13)

out_png = os.path.join(OUT_DIR, "ipa_compare_avoid_step_vs_floor.png")
plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

# Side-by-side numeric diff (floor - avoid_step) per BS
merged = avoid_df[["P%"]].copy()
for bs in BATCH_SIZES:
    col = f"IPA_Avg_{bs}"
    if col in avoid_df.columns and col in floor_df.columns:
        merged[f"avoid_{bs}"] = avoid_df[col].values
        merged[f"floor_{bs}"] = floor_df[col].values
        merged[f"diff_{bs}"]  = floor_df[col].values - avoid_df[col].values
print(merged.to_string(index=False))

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor\ipa_compare_avoid_step_vs_floor.png
   P%  avoid_64  floor_64   diff_64  avoid_1024  floor_1024  diff_1024  avoid_60000  floor_60000  diff_60000
  0.0  0.063152  0.050473 -0.012679    0.086264    0.066880  -0.019383     0.089746     0.073994   -0.015752
 10.0  0.058934  0.046512 -0.012421    0.082426    0.062390  -0.020036     0.085527     0.068520   -0.017007
 20.0  0.052015  0.043107 -0.008908    0.075388    0.056521  -0.018867     0.078179     0.063717   -0.014463
 30.0  0.049861  0.039116 -0.010745    0.066686    0.051599  -0.015087     0.069226     0.059506   -0.009720
 40.0  0.042452  0.034504 -0.007948    0.057929    0.046217  -0.011712     0.061856     0.051140   -0.010717
 50.0  0.036799  0.029249 -0.007550    0.048369    0.039093  -0.009275     0.051171     0.044633   -0.006538
 60.0  0.026447  0.024239 -0.002208    0.040303    0.033054  -0.007249 